# A large scale analysis of conversation data across myriad spoken corpora

In [1]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [2]:
bse = .002200

In [3]:
DATA_LOC = '../data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join(DATA_LOC, 'web_vis')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join(DATA_LOC, 'resids.parquet')
print(os.path.exists(FINAL_DOC))

True


In [4]:
# other global variables
turn_no_col = 'turn_delta'

In [5]:
def grab_all_files(PATH, file_type='.csv'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

In [6]:
# to rename the corpus correctly . . . 
def rename(x):
    result = x
    if x.startswith('se'):
        result = 'CORAAL'
        
    if x.startswith('call'):
        result = x.split('-')[0]
    
    if x.startswith('MICASE'):
        result = 'MICASE'
    
    if x.startswith('instruction'):
        result = x.split('-')[-1]
    
    if x.startswith('CABNC'):
        result = 'CABNC'
    
    return result

#### Load data

In [7]:
df_ = pd.read_parquet(FINAL_DOC)

In [8]:
# df_['corpus'] = [rename(co) for co in tqdm(df_['corpus'])]

In [9]:
df_['self'] = df_['self'].replace({True: 'self', False: 'other'})
df_['self'].loc[df_['null']] = 'null-model'
df_['self'].value_counts()

self
self          12306457
other         10926676
null-model     2192809
Name: count, dtype: int64

In [10]:
df_.head()

,Hxy,nx,ny,turn_delta,self,x_speaker,dyad,corpus,conversation_id,x_turn_id,null,groups,sample_delta,resid
0,6.344819,20.0,6.0,1,other,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-GCSAusE-GCSAusE-01-6,1,0.355428
1,6.000488,20.0,8.0,2,other,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-GCSAusE-GCSAusE-01-6,2,0.035463
2,5.521488,20.0,13.0,4,other,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-GCSAusE-GCSAusE-01-6,3,-0.382625
3,6.057978,20.0,6.0,6,other,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-GCSAusE-GCSAusE-01-6,4,0.068587
4,6.198739,20.0,6.0,7,other,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-GCSAusE-GCSAusE-01-6,5,0.209349


## Vis & Other Tests

In [11]:
import seaborn as sns
import plotly.tools as tls
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

In [12]:
names = ['null', 'other', 'self']
main_line_colors = ['#3e474d', '#1f77b4','#ff7f0e']
ribbon_colors = ['rgba(62, 71, 77, 0.2)', 'rgba(31, 119, 180, 0.2)', 'rgba(255, 127, 14, 0.2)']

In [13]:
df_ = df_.loc[
    (df_[turn_no_col] > 0)
    & (df_[turn_no_col] <= 30)
]

df = df_

In [14]:
df_ = df[['self', turn_no_col, 'resid']].groupby(by=['self', turn_no_col]).agg('mean').reset_index()
df_['ste'] = df[['self', turn_no_col, 'resid']].groupby(by=['self', turn_no_col]).agg('std').reset_index()['resid']
df_['nk'] = df[['self', turn_no_col, 'resid']].groupby(by=['self', turn_no_col]).agg('count').reset_index()['resid']
df_['SE'] = df_['ste'] / np.sqrt(df_['nk'])

In [20]:
fig2 = go.Figure()

In [21]:
# self
sel = df_['self'] == 'self'
fig2.add_trace(
    go.Scatter(
        y=df_['resid'].loc[sel].values,
        x=df_[turn_no_col].loc[sel].values,
        mode='lines',
        name='self',
        showlegend=True,
        legendgroup='self',
        line = dict(
            color='orange', 
            # width=4, 
            # dash='dash'
        )
    )
)

# upper bounds for ribbon
fig2.add_trace(
    go.Scatter(
        x=df_[turn_no_col].loc[sel].values,
        y=df_['resid'].loc[sel].values + (2*df_['SE'].loc[sel].values),
        mode='lines',
        line=dict(color='orange',width=0),
        name='self_upper_bound',
        showlegend=False,
        legendgroup='self',
        fill='tonexty',
        fillcolor='rgba(255, 165, 0, 0.2)',
    ), 
    # row=row, col=col
)

# lower bounds for ribbon
fig2.add_trace(
    go.Scatter(
        x=df_[turn_no_col].loc[sel].values,
        y=df_['resid'].loc[sel].values - (2*df_['SE'].loc[sel].values),
        mode='lines',
        line=dict(color='orange',width=0),
        name='self_lower_bound',
        showlegend=False,
        legendgroup='self',
        fill='tonexty',
        fillcolor='rgba(255, 165, 0, 0.2)',
    ), 
    # row=row, col=col
)

####################################
# other
####################################
sel = df_['self']  == 'other'
fig2.add_trace(
    go.Scatter(
        y=df_['resid'].loc[sel].values,
        x=df_[turn_no_col].loc[sel].values,
        mode='lines',
        name='other',
        showlegend=True,
        legendgroup='other',
        line = dict(
            color='royalblue', 
            # width=4, 
            # dash='dash'
        )
    )
)

# upper bounds for ribbon
fig2.add_trace(
    go.Scatter(
        x=df_[turn_no_col].loc[sel].values,
        y=df_['resid'].loc[sel].values + (2*df_['SE'].loc[sel].values),
        mode='lines',
        line=dict(color='orange',width=0),
        name='other_upper_bound',
        showlegend=False,
        legendgroup='other',
        fill='tonexty',
        fillcolor='rgba(65, 105, 225, 0.2)',
    ), 
    # row=row, col=col
)

# lower bounds for ribbon
fig2.add_trace(
    go.Scatter(
        x=df_[turn_no_col].loc[sel].values,
        y=df_['resid'].loc[sel].values - (2*df_['SE'].loc[sel].values),
        mode='lines',
        line=dict(color='orange',width=0),
        name='other_lower_bound',
        showlegend=False,
        legendgroup='other',
        fill='tonexty',
        fillcolor='rgba(65, 105, 225, 0.2)',
    ), 
    # row=row, col=col
)

####################################
# null
####################################
sel = df_['self'] == 'null-model'
fig2.add_trace(
    go.Scatter(
        y=df_['resid'].loc[sel].values,
        x=df_[turn_no_col].loc[sel].values,
        mode='lines',
        name='null-model',
        showlegend=True,
        legendgroup='null-model',
        line = dict(
            color='#3e474d',
            # width=4,
            # dash='dash'
        )
    )
)

# upper bounds for ribbon
fig2.add_trace(
    go.Scatter(
        x=df_[turn_no_col].loc[sel].values,
        y=df_['resid'].loc[sel].values + (2*df_['SE'].loc[sel].values),
        mode='lines',
        line=dict(color='orange',width=0),
        name='null-model_upper_bound',
        showlegend=False,
        legendgroup='null-model',
        fill='tonexty',
        fillcolor='rgba(62, 71, 77, 0.2)',
    ),
    # row=row, col=col
)

# lower bounds for ribbon
fig2.add_trace(
    go.Scatter(
        x=df_[turn_no_col].loc[sel].values,
        y=df_['resid'].loc[sel].values - (2*df_['SE'].loc[sel].values),
        mode='lines',
        line=dict(color='orange',width=0),
        name='null-model_lower_bound',
        showlegend=False,
        legendgroup='null-model',
        fill='tonexty',
        fillcolor='rgba(62, 71, 77, 0.2)',
    ),
    # row=row, col=col
)

In [22]:
fig2.update_layout(template='xgridoff')
fig2.show() 

In [23]:
fig2.write_html(os.path.join(VIS_PATH, 'entropy-delta.html'))

## Vis & Other Tests

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
names = ['other', 'self']
main_line_colors = ['#1f77b4','#ff7f0e']
ribbon_colors = ['rgba(31, 119, 180, 0.2)', 'rgba(255, 127, 14, 0.2)']

#### Initial plots

In [ ]:
# sns.lineplot(
#     data=df_.loc[
#         ~df_['self']
#         & (df_[turn_no_col] > 0)
#         & (df_[turn_no_col] < 40)
#     ], 
#     y='resid', 
#     x=turn_no_col, 
# )
# plt.show()

In [ ]:
# sns.lineplot(
#     data=df_.loc[
#         df_['self'] 
#         & (df_[turn_no_col] > 0)
#         & (df_[turn_no_col] < 40)
#     ], 
#     y='resid', 
#     x=turn_no_col, 
#     color='orange'
# )
# plt.show()

In [ ]:
ax = sns.lineplot(
    data=df_.loc[
        (df_[turn_no_col] > 0)
        & (df_[turn_no_col] < 40)
    ], 
    y='resid', 
    x=turn_no_col, 
    hue='self',
    legend=False
)
plt.show()

### Creating an interactive plot using px to start, and then converting to GO :)

In [ ]:
import plotly.tools as tls
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
fig = tls.mpl_to_plotly(ax.figure)

In [ ]:
new_fig = go.Figure()

In [ ]:
for i,trace in enumerate(fig.data):
    trace['name'] = names[i]
    # print(trace)
    trace['showlegend'] = True
    trace['line']['color'] = main_line_colors[i]
    trace['legendgroup'] = names[i]
    # trace['legendgrouptitle'] = names[i]
    new_fig.add_trace(trace)
    
    # upper bounds for ribbon
    new_fig.add_trace(
        go.Scatter(
            x=trace['x'],
            y=(np.array(trace['y']) + (bse*2)).tolist(),
            mode='lines',
            line=dict(color=main_line_colors[i],width=0),
            name=names[i]+'_upper_bound',
            showlegend=False,
            legendgroup=names[i]
        )
    )
    
    
    # lower bounds for ribbon
    new_fig.add_trace(
        go.Scatter(
            x=trace['x'],
            y=(np.array(trace['y']) - (bse*2)).tolist(),
            mode='lines',
            line=dict(color=main_line_colors[i],width=0),
            name=names[i]+'_lower_bound',
            fill='tonexty',
            fillcolor=ribbon_colors[i],
            showlegend=False,
            legendgroup=names[i]
        )
    )
    
# new_fig.update_layout(showlegend=True)
new_fig.update_layout(template='plotly')
new_fig.show()  

In [ ]:
new_fig.write_html(os.path.join(VIS_PATH, 'entropy-delta.html'))

In [ ]:
new_fig.to_image(os.path.join(VIS_PATH, 'entropy-delta.png'), scale=5)